<a href="https://colab.research.google.com/github/AmanGupta3995377/Celebal-DataScience-Internship/blob/main/Week-7/Week7_Aman_Gupta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 Assignment
# Document Question Answering System using RAG (Retrieval-Augmented Generation

## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) based Question Answering System.

The system allows users to upload custom documents such as PDFs and ask questions related to their content. Instead of relying solely on the language model's internal knowledge, the system retrieves relevant information from the document and generates context-aware answers.

The pipeline consists of:

1. Document Ingestion
2. Text Chunking
3. Embedding Generation
4. Vector Database Creation
5. Similarity Search
6. Context Retrieval
7. Answer Generation using LLM

## Objectives

- Understand the concept of Retrieval-Augmented Generation (RAG)
- Build an end-to-end document question answering system
- Convert unstructured text into vector embeddings
- Store embeddings in a vector database
- Retrieve relevant document chunks
- Generate grounded answers using retrieved context
- Evaluate retrieval quality using sample queries

## Install Libraries

In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q torch

## Import Libraries

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline
from google.colab import files
import pandas as pd

/tmp/ipykernel_4089/236962883.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## PDF Upload

In [3]:
uploaded = files.upload()

Saving Notes.pdf to Notes.pdf


## Loading PDF and priviewing extracted text

In [5]:
loader = PyPDFLoader("Notes.pdf")
documents = loader.load()

print("Total Pages:", len(documents))

print(documents[0].page_content[:1000])

Total Pages: 93
Essentials of Cyber Security (CSE336)
Module – 2
(Cyber Law & Policy)


## Chunking the text

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 148


## Creation of Embedding Model and Checking Embedding Dimension

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding Model Loaded Successfully")

sample_embedding = embedding_model.embed_query(
    "What is Retrieval Augmented Generation?"
)
print("Embedding Dimension:", len(sample_embedding))

/tmp/ipykernel_4089/3972275974.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully
Embedding Dimension: 384


## Creating FAISS Vector Store

In [11]:
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)
print("FAISS Vector Database Created Successfully")

FAISS Vector Database Created Successfully


## Retrieval Function

In [12]:
def retrieve_context(question):

    docs = vector_db.similarity_search(
        question,
        k=3
    )
    return docs

## Test Retrieval

In [13]:
question = "What is the main topic of the document?"
retrieved_docs = retrieve_context(question)

for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i+1}")
    print("-"*50)
    print(doc.page_content[:500])


Chunk 1
--------------------------------------------------
• Musical works – songs, compositions.
• Artistic works – paintings, drawings, photographs.
• Dramatic works – plays, scripts.
• Cinematographic films – movies, videos.
• Sound recordings – music albums, audiobooks.
• Software – treated as “literary work.”

Chunk 2
--------------------------------------------------
Important Sections of IT Act
Section 43 – Penalty for unauthorized access, data theft, spreading viruses.
Section 65 – Tampering with computer source code.
Section 66 – Hacking.
Section 66C – Identity theft (stealing password, digital signature).
Section 66D – Cheating by impersonation (phishing, fraud).
Section 67 – Publishing obscene content online.
Section 72 – Breach of confidentiality and privacy.
Example:
• If a hacker steals ATM card details → punished under Section 66C (identity theft).

Chunk 3
--------------------------------------------------
• Section 66E – Violation of privacy (capturing/sharing private

## Load FLAN-T5

In [21]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM Loaded Successfully")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

LLM Loaded Successfully


## Context Retrieval Demonstration

In [22]:
def answer_question(question):

    docs = retrieve_context(question)

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
Answer the question based only on the provided context.

Context:
{context}

Question:
{question}

Short Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=50,
        do_sample=False
    )

    generated_text = result[0]["generated_text"]

    answer = generated_text.split("Short Answer:")[-1].strip()

    return answer

##  Query Processing and Context Retrieval

In [26]:
question = "What is Cyber Security?"

docs = retrieve_context(question)

print("Question:", question)

print("\nRetrieved Context:\n")

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}")
    print("-"*50)
    print(doc.page_content[:500])

Question: What is Cyber Security?

Retrieved Context:


Chunk 1
--------------------------------------------------
Cybersecurity Management Concepts
What is Cybersecurity Management?
• Cybersecurity management is the process of planning, controlling, and monitoring security
policies and practices to protect an organization’s data, systems, and networks from cyber
threats.
A threat in cybersecurity is anything that can cause harm to your computer,device, data, or online
accounts.
• It’s not just about installing antivirus or firewalls — it’s about managing people, processes, and
technology together.

Chunk 2
--------------------------------------------------
• Functions describe step-by-step tasks to protect, detect, respond, and recover from cyber threats.
• Cybersecurity is not just technical – it is also about people, processes, and management.

Chunk 3
--------------------------------------------------
technology together.
Example - Imagine you are the security manager of a bank. Yo

## Validation Questions

In [27]:
questions = [

    "What is Cyber Security?",

    "What are the objectives of Cyber Security?",

    "What is Cyber Law?",

    "What is the IT Act 2000?",

    "What are common cyber threats?"
]

## Validation Logs

In [29]:
for q in questions:
    docs = retrieve_context(q)

    print("\nQuestion:", q)
    print("\nTop Retrieved Context:\n")
    print(docs[0].page_content[:500])
    print("\n" + "="*80)


Question: What is Cyber Security?

Top Retrieved Context:

Cybersecurity Management Concepts
What is Cybersecurity Management?
• Cybersecurity management is the process of planning, controlling, and monitoring security
policies and practices to protect an organization’s data, systems, and networks from cyber
threats.
A threat in cybersecurity is anything that can cause harm to your computer,device, data, or online
accounts.
• It’s not just about installing antivirus or firewalls — it’s about managing people, processes, and
technology together.


Question: What are the objectives of Cyber Security?

Top Retrieved Context:

management to provide strategic direction, ensure objectives are achieved, manage risk,
and verify that resources are used responsibly for information security.”
In Simple Words:
It is how leaders make sure cybersecurity is working properly and supports the business.


Question: What is Cyber Law?

Top Retrieved Context:

• The Act gives legal recognition to digital

## System Metrics Report

In [30]:
metrics = {
    "Total Pages":93,
    "Total Chunks":148,

    "Chunk Size":500,
    "Chunk Overlap":50,

    "Embedding Model":"all-MiniLM-L6-v2",
    "Embedding Dimension":384,

    "Vector Store":"FAISS",
    "Top K Retrieval":3
}
metrics_df = pd.DataFrame(
    metrics.items(),
    columns=["Metric","Value"]
)
metrics_df

,Metric,Value
0,Total Pages,93
1,Total Chunks,148
2,Chunk Size,500
3,Chunk Overlap,50
4,Embedding Model,all-MiniLM-L6-v2
5,Embedding Dimension,384
6,Vector Store,FAISS
7,Top K Retrieval,3


## Optimization Experiment

In [31]:
for size in [300,500]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=50
    )
    temp_chunks = splitter.split_documents(
        documents
    )
    print(
        f"Chunk Size {size}: {len(temp_chunks)} chunks"
    )

Chunk Size 300: 226 chunks
Chunk Size 500: 148 chunks


# Conclusion

This project successfully implemented a Retrieval-Augmented Generation (RAG) based Document Question Answering System using a Cyber Security PDF document.
The system performed document ingestion, text chunking, embedding generation, vector database creation using FAISS, and similarity-based retrieval. Relevant document chunks were successfully retrieved for user queries, demonstrating effective semantic search and context retrieval.

### Key Outcomes
- Loaded and processed a 93-page PDF document.
- Generated 148 text chunks.
- Created embeddings using all-MiniLM-L6-v2.
- Stored embeddings in a FAISS vector database.
- Retrieved relevant context for user queries.
- Performed validation testing and optimization experiments.

Overall, this project provided practical experience in building a RAG pipeline and understanding how retrieval techniques improve question answering over custom documents.